In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-72B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-72B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
#     temperature=0.0,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|████████████████████████████████████████████████| 37/37 [01:23<00:00,  2.26s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [2]:
import pandas as pd 
import re
import torch

# Load data
df = pd.read_csv(paths.DATA / "MedPAIR_1300_Sentence.csv")
print("Columns in dataset:")
print(df.columns.tolist())

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21']


In [3]:
import os
import re
import ast
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from time import time as timer

# === Prepare Output File ===
output_path = paths.DATA / "Qwen72B_annotated_MedPAIR_relevancy.csv"

# Create output columns
output_cols = df.columns.tolist() + ['Qwen72B_answer', 'Qwen72B_raw_response'] + [f"label_{i+1}" for i in range(21)]
sentence_cols = [f'sentence_{i}' for i in range(1, 22)]

# === Check for Existing Progress ===
if os.path.exists(output_path):
    print(f"📂 Found existing file: {output_path}")
    existing_df = pd.read_csv(output_path)
    processed_rows = len(existing_df)
    print(f"✅ Already processed: {processed_rows} rows")
    
    # Load existing results
    results = existing_df.values.tolist()
    start_idx = processed_rows
    print(f"🔄 Resuming from row {start_idx + 1}")
else:
    print(f"📝 No existing file found. Starting fresh...")
    results = []
    start_idx = 0

# === Process Each Row ===
total_rows = len(df)
remaining_rows = total_rows - start_idx

print(f"\n{'='*60}")
print(f"Total rows: {total_rows}")
print(f"Already processed: {start_idx}")
print(f"Remaining to process: {remaining_rows}")
print(f"{'='*60}\n")

if remaining_rows == 0:
    print("✅ All rows already processed!")
else:
    print(f"Processing rows {start_idx + 1} to {total_rows}...\n")

for idx in range(start_idx, total_rows):
    print(f"\n{'='*60}")
    print(f"Processing Row {idx + 1}/{total_rows} (Progress: {idx - start_idx + 1}/{remaining_rows})")
    print(f"{'='*60}")
    
    try:
        # === Extract Sentences and Options ===
        sentences = [df.loc[idx, col] for col in sentence_cols if pd.notna(df.loc[idx, col])]
        num_sentences = len(sentences)
        
        if num_sentences == 0:
            print(f"⚠️ No sentences found for row {idx}, skipping...")
            # Add empty result to maintain row alignment
            result_row = df.loc[idx].tolist() + [None, "No sentences found"] + [None] * 21
            results.append(result_row)
            continue
        
        formatted_sentences = "\n".join(f"{j+1}. {s.strip()}" for j, s in enumerate(sentences))
        options = df.loc[idx, "question_options_x"]
        
        # === Display Memory Info ===
        if torch.cuda.is_available():
            free_mem = torch.cuda.memory_reserved(0) - torch.cuda.memory_allocated(0)
            print(f"🖥️ CUDA Free Memory: {free_mem / (1024 ** 2):.2f} MB")
        
        # === Format the Prompt ===
        prompt = f"""You are given a list of sentences from a clinical vignette and a multiple-choice clinical question.

Your task is twofold:
(1) Select the most appropriate answer from the given options.
(2) Label *each* of the {num_sentences} sentences as either [High Relevance], [Low Relevance], or [Irrelevant], based on its contribution to answering the question.

Definitions:
[High Relevance]: Sentences that directly support the correct answer with essential clinical information (e.g., diagnosis, key test results).
[Low Relevance]: Sentences that provide useful context or background but are not critical to answering.
[Irrelevant]: Sentences unrelated to the question or not useful for reasoning.

Question and Options:
{options}

Sentences:
{formatted_sentences}

Please provide your response in the following format:
Answer: [Your selected option letter]
Sentence 1: [High Relevance/Low Relevance/Irrelevant]
Sentence 2: [High Relevance/Low Relevance/Irrelevant]
...
(Continue for all {num_sentences} sentences)"""
        
        # === Generate Response Using Chat Template ===
        messages = [{"role": "user", "content": prompt}]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        print(f"🔍 Input Token Count: {len(model_inputs['input_ids'][0])}")
        
        # === Time the Generation ===
        start_time = timer()
        
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=1024,
                do_sample=False
            )
        
        end_time = timer()
        print(f"⏱️ Generation Time: {end_time - start_time:.2f} seconds")
        
        # === Decode Response ===
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # === Display Raw Response ===
        print(f"\n{'='*60}")
        print("Raw Model Response:")
        print(f"{'='*60}")
        print(response)
        print(f"{'='*60}")
        
        # === Extract Answer ===
        answer_match = re.search(r"Answer:\s*([A-J])", response, re.IGNORECASE)
        extracted_answer = answer_match.group(1).upper() if answer_match else None
        
        # === Extract Sentence Labels ===
        labels = [None] * 21  # Initialize all labels as None
        
        for sent_num in range(1, num_sentences + 1):
            # Look for patterns like "Sentence 1: [High Relevance]" or "1. High Relevance"
            pattern = rf"Sentence\s*{sent_num}:\s*\[?(High Relevance|Low Relevance|Irrelevant)\]?"
            match = re.search(pattern, response, re.IGNORECASE)
            
            if match:
                labels[sent_num - 1] = match.group(1)
            else:
                # Try alternative pattern
                pattern2 = rf"{sent_num}\.\s*\[?(High Relevance|Low Relevance|Irrelevant)\]?"
                match2 = re.search(pattern2, response, re.IGNORECASE)
                if match2:
                    labels[sent_num - 1] = match2.group(1)
        
        print(f"\n📊 Extracted Answer: {extracted_answer}")
        print(f"📊 Extracted Labels: {labels[:num_sentences]}")
        
        # === Build Result Row ===
        result_row = df.loc[idx].tolist() + [extracted_answer, response] + labels
        results.append(result_row)
        
        # === Save Incrementally Every 10 Rows ===
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results, columns=output_cols)
            temp_df.to_csv(output_path, index=False)
            print(f"\n💾 Saved progress: {idx + 1} rows completed")
        
    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        import traceback
        traceback.print_exc()
        # Add empty result to maintain row alignment
        result_row = df.loc[idx].tolist() + [None, str(e)] + [None] * 21
        results.append(result_row)
    
    # === Clear CUDA Cache ===
    torch.cuda.empty_cache()

# === Final Save ===
print("\n" + "="*60)
print("Processing Complete!")
print("="*60)

final_df = pd.DataFrame(results, columns=output_cols)
final_df.to_csv(output_path, index=False)
print(f"✅ Final results saved to: {output_path}")
print(f"📊 Total rows processed: {len(results)}")
print(f"📊 New rows added in this run: {len(results) - start_idx}")

📂 Found existing file: Qwen72B_annotated_MedPAIR_relevancy.csv
✅ Already processed: 1080 rows
🔄 Resuming from row 1081

Total rows: 1300
Already processed: 1080
Remaining to process: 220

Processing rows 1081 to 1300...


Processing Row 1081/1300 (Progress: 1/220)
🖥️ CUDA Free Memory: 43.91 MB
🔍 Input Token Count: 831
⏱️ Generation Time: 54.53 seconds

Raw Model Response:
Answer: C

Sentence 1: [High Relevance] - Provides the patient's age, gender, and the primary symptom, which is crucial for the differential diagnosis.
Sentence 2: [Low Relevance] - Mentions the timing of the rash relative to the COVID-19 vaccine, which could be relevant for ruling out vaccine-related conditions but is not critical for the final diagnosis.
Sentence 3: [Low Relevance] - Denial of specific symptoms can help rule out certain conditions but is not critical for the diagnosis.
Sentence 4: [Low Relevance] - Background medical history that provides context but is not directly relevant to the diagnosis.
Senten

⏱️ Generation Time: 36.74 seconds

Raw Model Response:
Answer: C. Amoxicillin and clavulanate

Sentence 1: Low Relevance
Sentence 2: High Relevance
Sentence 3: High Relevance
Sentence 4: Low Relevance
Sentence 5: Irrelevant
Sentence 6: High Relevance
Sentence 7: High Relevance
Sentence 8: High Relevance

Explanation:
- **Sentence 1**: Provides the setting and patient's chief complaint, which is important for context but not critical for the specific management decision.
- **Sentence 2**: Directly relevant as it describes the symptom (leg pain) that led to the presentation.
- **Sentence 3**: Highly relevant as it provides the history of a potential source of infection (cat bite), which is crucial for understanding the etiology.
- **Sentence 4**: Provides context about the initial wound care, which is less critical for the current management decision.
- **Sentence 5**: Irrelevant as it does not contribute to the decision-making process.
- **Sentence 6**: Highly relevant as it includes vi

⏱️ Generation Time: 39.52 seconds

Raw Model Response:
Answer: C

Sentence 1: [Low Relevance]
Sentence 2: [Low Relevance]
Sentence 3: [Low Relevance]
Sentence 4: [High Relevance]
Sentence 5: [Low Relevance]
Sentence 6: [Low Relevance]
Sentence 7: [Low Relevance]
Sentence 8: [Low Relevance]
Sentence 9: [High Relevance]
Sentence 10: [High Relevance]
Sentence 11: [High Relevance]
Sentence 12: [High Relevance]
Sentence 13: [High Relevance]
Sentence 14: [High Relevance]
Sentence 15: [High Relevance]
Sentence 16: [Low Relevance]

Explanation:
- **High Relevance**: Sentences 4, 9, 10, 11, 12, 13, 14, and 15 provide critical information about the clinical presentation, progression, and specific findings that suggest a viral etiology, which supports the choice of obtaining an aqueous fluid sample for PCR and treating with antiviral agents.
- **Low Relevance**: Sentences 1, 2, 3, 5, 6, 7, 8, and 16 provide context and background information but are not essential for making the decision to procee

⏱️ Generation Time: 27.72 seconds

Raw Model Response:
Answer: B
Sentence 1: [High Relevance] - This sentence provides the patient's age, health status, and the primary presenting symptom, which is crucial for understanding the clinical scenario.
Sentence 2: [High Relevance] - This sentence describes the physical appearance of the lesion, which is important for making a diagnosis and deciding on the next steps.
Sentence 3: [Low Relevance] - While it is good to know that the patient has normal vision, this information is not critical for deciding the next step in management.
Sentence 4: [Low Relevance] - Normal ocular motility is useful to rule out certain conditions but does not directly influence the decision for the next step.
Sentence 5: [Low Relevance] - Normal findings in the anterior and posterior segments provide reassurance but do not directly impact the choice of the next action.
Sentence 6: [Irrelevant] - This sentence suggests a treatment option, which is one of the choices 

⏱️ Generation Time: 37.26 seconds

Raw Model Response:
Answer: B. Adjusting her medication regimen

Sentence 1: [Low Relevance] - Provides the patient's age and chief complaint, which is important for context but not critical for the decision-making process.
Sentence 2: [High Relevance] - Describes symptoms of orthostatic hypotension, which is crucial for understanding the patient's condition and potential causes.
Sentence 3: [Low Relevance] - Provides background medical history, which is useful for context but not directly relevant to the immediate management.
Sentence 4: [High Relevance] - Lists current medications, which are directly relevant to the management decision, especially considering the possibility of medication-induced side effects.
Sentence 5: [High Relevance] - Indicates a slow heart rate, which could be contributing to the symptoms and is relevant to the management.
Sentence 6: [High Relevance] - Shows significant orthostatic hypotension, which is a key finding support

...

⏱️ Generation Time: 44.27 seconds

Raw Model Response:
Answer: C. Propranolol

Sentence 1: [High Relevance] - Introduces the main symptom (hand shaking) which is crucial for determining the type of tremor and appropriate treatment.
Sentence 2: [High Relevance] - Provides more detail about the nature of the tremor, indicating it occurs during specific activities, which is characteristic of essential tremor.
Sentence 3: [High Relevance] - Describes the pattern of the tremor improving over the day, which is consistent with essential tremor.
Sentence 4: [Low Relevance] - Provides additional medical history that is not directly related to the tremor but may influence overall management.
Sentence 5: [Low Relevance] - Lists current medications, which is important for avoiding drug interactions but not directly relevant to the choice of treatment for the tremor.
Sentence 6: [Low Relevance] - Information about smoking and alcohol use, which could potentially exacerbate tremors but is not direct

⏱️ Generation Time: 26.80 seconds

Raw Model Response:
Answer: H
Sentence 1: Low Relevance
Sentence 2: High Relevance
Sentence 3: High Relevance
Sentence 4: Low Relevance
Sentence 5: High Relevance

Explanation:
- **Sentence 1**: Provides context about the patient's age and the reason for the visit, which is important but not critical to determining the next step in management.
- **Sentence 2**: Describes symptoms that are highly relevant to the suspected condition (likely anabolic steroid use), making it crucial for the decision-making process.
- **Sentence 3**: Further elaborates on the patient's mental state and behavior, which supports the need for careful management and potential tapering of any suspected substance.
- **Sentence 4**: Provides context about the patient's lifestyle, which is useful but not directly critical to the immediate management decision.
- **Sentence 5**: Physical findings (muscular build, acne) are highly relevant as they support the suspicion of anabolic st

⏱️ Generation Time: 55.61 seconds

Raw Model Response:
Answer: H. Obtain a chest CT with contrast

Sentence 1: [Low Relevance] - Provides demographic and general presentation, but not specific to the next step in management.
Sentence 2: [Low Relevance] - Describes symptoms that are part of the clinical picture but not critical for the immediate next step.
Sentence 3: [High Relevance] - Swollen lymph nodes suggest a possible underlying condition that needs further investigation.
Sentence 4: [High Relevance] - Describes the progression of symptoms, particularly the facial swelling, which is relevant to the differential diagnosis.
Sentence 5: [Low Relevance] - Denial of other symptoms helps narrow the differential but is not critical for the next step.
Sentence 6: [Low Relevance] - Information about eating, drinking, and urine output provides general health status but is not critical for the next step.
Sentence 7: [Low Relevance] - Medication and allergy history is important for overall c

⏱️ Generation Time: 21.16 seconds

Raw Model Response:
Answer: D
Sentence 1: [High Relevance] - The patient's age and gender (20-year-old female) are important for considering conditions such as pregnancy.
Sentence 2: [High Relevance] - The presence of blood in the vomit could indicate a serious condition, but it also raises the possibility of other causes, including those related to pregnancy.
Sentence 3: [Low Relevance] - Stress can contribute to symptoms like nausea and vomiting, but it is not specific enough to guide the initial test choice.
Sentence 4: [Low Relevance] - High caffeine intake can cause gastrointestinal symptoms, but it is less likely to be the primary cause of bloody vomit.
Sentence 5: [Low Relevance] - Normal vital signs provide context but do not specifically direct the choice of the initial test.

📊 Extracted Answer: D
📊 Extracted Labels: ['High Relevance', 'High Relevance', 'Low Relevance', 'Low Relevance', 'Low Relevance']

Processing Row 1295/1300 (Progress: 2

⏱️ Generation Time: 26.66 seconds

Raw Model Response:
Answer: E
Sentence 1: Low Relevance
Sentence 2: High Relevance
Sentence 3: Low Relevance
Sentence 4: Irrelevant
Sentence 5: Low Relevance

Explanation:
- **Sentence 1**: Provides general information about the patient's age and initial symptoms, which are somewhat relevant but not specific enough to point towards a particular diagnosis.
- **Sentence 2**: Highly relevant as it describes the patient's history of mood swings, including periods of increased productivity and elevated mood followed by depressive symptoms, which are characteristic of bipolar disorder.
- **Sentence 3**: Low relevance as it provides information about alcohol consumption, which could be a factor but is not central to diagnosing bipolar disorder.
- **Sentence 4**: Irrelevant because normal physical examination findings do not contribute to the psychiatric diagnosis.
- **Sentence 5**: Low relevance as it describes the patient's appearance and some cognitive sym

In [6]:
import pandas as pd

Qwen72B_Labels = pd.read_csv(paths.DATA / "Qwen72B_annotated_MedPAIR_relevancy.csv")
Qwen72B_Labels.columns

original_path = paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv"
trainee = pd.read_csv(original_path)
trainee.columns
# physician.loc[:, "q2":"q22"]

# Rename label_1–label_20 → q_1–q_20
rename_dict = {f"label_{i}": f"q{i}" for i in range(1, 21)}
Qwen72B_Labels = Qwen72B_Labels.rename(columns=rename_dict)

# Verify
Qwen72B_Labels.columns

Index(['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High',
       'question_options_x', 'answer_corr', 'ID', 'centaur_question',
       'sentence_number', 'answer', 'data_source', 'step1_excerpts',
       'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2',
       'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7',
       'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12',
       'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16',
       'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20',
       'sentence_21', 'Qwen72B_answer', 'Qwen72B_raw_response', 'q1', 'q2',
       'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13',
       'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21'],
      dtype='object')

In [7]:
df = Qwen72B_Labels.copy()

# -------------------------
# 1. Empty rate calculation
# -------------------------
# Define empty as NaN or empty/whitespace-only string
is_empty = df['Qwen72B_answer'].isna() | (df['Qwen72B_answer'].astype(str).str.strip() == "")
empty_rate = is_empty.mean()

# -------------------------
# 2. Accuracy calculation
# -------------------------
# Compute accuracy only on non-empty predictions
valid_mask = ~is_empty

# Optional normalization for robust matching
pred = df.loc[valid_mask, 'Qwen72B_answer'].astype(str).str.strip().str.lower()
gt = df.loc[valid_mask, 'answer'].astype(str).str.strip().str.lower()

accuracy = (pred == gt).mean()

print(f"Qwen72B_answer empty rate: {empty_rate:.4f}")
print(f"Accuracy on non-empty predictions: {accuracy:.4f}")

Qwen72B_answer empty rate: 0.0038
Accuracy on non-empty predictions: 0.5019


In [8]:
cols_trainee = [f"q{i}" for i in range(1, 21)]

def transform_trainee(value):
    if pd.isna(value):
        return value
    
    val_str = str(value).strip().upper()
    
    # Check if numeric (including floats stored as strings)
    try:
        float(value)
        return "high relevance"
    except ValueError:
        pass
    
    if val_str == "REMOVED":
        return "low relevance"
    else:
        return value

trainee[cols_trainee] = trainee[cols_trainee].apply(lambda col: col.map(transform_trainee))

In [9]:
cols_Qwen72B = [f"q{i}" for i in range(1, 21)]

def transform_Qwen72B(value):
    if pd.isna(value):
        return value
    val = str(value).strip().lower()
    if val == "not applicable":
        return ""
    elif val == "not relevant":
        return "low relevance"
    else:
        return value

Qwen72B_Labels[cols_Qwen72B] = Qwen72B_Labels[cols_Qwen72B].map(transform_Qwen72B)

In [12]:
Qwen72B_Labels = Qwen72B_Labels.sort_values(by='Origin')
for c in cols_Qwen72B:
    Qwen72B_Labels[c] = Qwen72B_Labels[c].apply(lambda x: x.lower() if isinstance(x, str) else x)
Qwen72B_Labels[["Origin"] + cols_Qwen72B].head(10)

,Origin,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,q11,q12,q13,q14,q15,q16,q17,q18,q19,q20
0,ID0002,high relevance,low relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,low relevance,...,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,low relevance,low relevance,irrelevant,low relevance,low relevance,high relevance,high relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,high relevance,high relevance,low relevance,high relevance,high relevance,low relevance,high relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,high relevance,high relevance,high relevance,low relevance,high relevance,high relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,low relevance,low relevance,irrelevant,irrelevant,high relevance,low relevance,high relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,high relevance,low relevance,low relevance,low relevance,low relevance,high relevance,low relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,low relevance,high relevance,high relevance,high relevance,low relevance,high relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,low relevance,high relevance,high relevance,high relevance,low relevance,low relevance,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,high relevance,high relevance,high relevance,low relevance,low relevance,high relevance,high relevance,high relevance,high relevance,...,low relevance,irrelevant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
def combine_high_relevance_sentences(row):
    """
    Combines sentences where corresponding q columns have 'high relevance'
    """
    high_relevance_sentences = []
    
    # Iterate through q1 to q20
    for i in range(1, 21):
        q_col = f'q{i}'
        sentence_col = f'sentence_{i}'
        
        # Check if q column contains 'high relevance' (case-insensitive)
        if pd.notna(row.get(q_col)) and pd.notna(row.get(sentence_col)):
            if 'high relevance' in str(row[q_col]).lower():
                high_relevance_sentences.append(str(row[sentence_col]))
    
    # Combine all high relevance sentences with a separator
    if high_relevance_sentences:
        return ' '.join(high_relevance_sentences)
    else:
        return None  # or return '' for empty string

# Apply the function to create the new column
Qwen72B_Labels['Qwen72B_High_Relevance'] = Qwen72B_Labels.apply(combine_high_relevance_sentences, axis=1)

# Optional: Check the results
print(f"Total rows with high relevance sentences: {Qwen72B_Labels['Qwen72B_High_Relevance'].notna().sum()}")
print("\nSample of the new column:")
print(Qwen72B_Labels[['Qwen72B_High_Relevance']].head(20))
Qwen72B_Labels.to_csv(paths.DATA / "Qwen72B_annotated_MedPAIR_relevancy.csv", index=False)

Total rows with high relevance sentences: 1294

Sample of the new column:
                               Qwen72B_High_Relevance
0   1. A woman in her 60s with a history of hyperl...
1   6. Chest auscultation is unremarkable, but a c...
2   1. A 72-year-old man presents to his primary c...
3   1. A woman in her 30s presented with multiple ...
4   5. Examination reveals pulsatile bleeding from...
5   1. A 3-year-old white girl is brought to the o...
6   1. A 60-year-old man with a history of IgA kap...
7   2. On examination, his visual acuity was count...
8   2. Computed tomography (CT) showed a right-sid...
9   1. A 74-year-old man with an ocular history of...
10  2. He presented with a 2-day history of painfu...
11  4. An endometrial biopsy specimen shows regula...
12  3. Physical examination reveals a prominent le...
13  1. A 47-year-old man comes to the physician 12...
14  5. Physical exam is notable only for minor pal...
15  1. A 46-year-old man presents with his wife to...
16  1. T

In [13]:
import numpy as np
import pandas as pd

# Columns to compare
cols_compare = [f"q{i}" for i in range(1, 20)]

# Create dictionary for fast lookup: Origin → trainee row
trainee_dict = {trainee.loc[i, "Origin"]: trainee.iloc[i] for i in range(len(trainee))}

def calculate_match(row_t, row_p):
    matches = 0
    total_valid = 0  # denominator: number of non-NaN labels

    for c in cols_compare:
        val_t = row_t[c]
        val_p = row_p[c]

        # Only consider this column if both values are not NaN
        if pd.notna(val_t) and pd.notna(val_p):
            total_valid += 1
            if val_t == val_p:
                matches += 1

    if total_valid == 0:
        return np.nan
    else:
        return f"{matches}/{total_valid}"  # numerator / denominator

# Apply only if Origins match
match_results = []
trainee_sources = []

for i in range(len(Qwen72B_Labels)):
    origin_p = Qwen72B_Labels.loc[i, "Origin"]
    if origin_p in trainee_dict:
        row_t = trainee_dict[origin_p]
        row_p = Qwen72B_Labels.iloc[i]
        match_results.append(calculate_match(row_t, row_p))
        trainee_sources.append(row_t.get("data_source_corr", np.nan))
    else:
        match_results.append(np.nan)
        trainee_sources.append(np.nan)

# Add results to DataFrame
Qwen72B_Labels["Match?"] = match_results
Qwen72B_Labels["data_source_corr_trainee"] = trainee_sources

# View results
Qwen72B_Labels[["Origin", "data_source_corr_trainee", "Match?"] + cols_compare].head(50)

,Origin,data_source_corr_trainee,Match?,q1,q2,q3,q4,q5,q6,q7,...,q10,q11,q12,q13,q14,q15,q16,q17,q18,q19
0,ID0002,jama,9/11,high relevance,low relevance,low relevance,high relevance,low relevance,low relevance,low relevance,...,low relevance,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,medxpert,1/7,low relevance,low relevance,irrelevant,low relevance,low relevance,high relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,medbullets,5/7,high relevance,high relevance,low relevance,high relevance,high relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,jama,5/8,high relevance,high relevance,high relevance,low relevance,high relevance,high relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,medxpert,3/7,low relevance,low relevance,irrelevant,irrelevant,high relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,mmlu,4/7,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,jama,7/10,high relevance,low relevance,low relevance,low relevance,low relevance,high relevance,low relevance,...,low relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,jama,3/7,low relevance,high relevance,high relevance,high relevance,low relevance,high relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,jama,5/6,low relevance,high relevance,high relevance,high relevance,low relevance,low relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,jama,8/12,high relevance,high relevance,high relevance,low relevance,low relevance,high relevance,high relevance,...,low relevance,low relevance,irrelevant,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
import numpy as np
import pandas as pd
from scipy import stats

# Function to convert 'Match?' to numeric percentage
def match_to_percent(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x) * 100
    if isinstance(x, str) and "/" in x:
        try:
            num, denom = x.split("/")
            return (float(num) / float(denom)) * 100
        except:
            return np.nan
    return np.nan

# Create a numeric column for Match? in percentage
Qwen72B_Labels["Match_percent"] = Qwen72B_Labels["Match?"].apply(match_to_percent)

# Group by data_source_corr_trainee
grouped = Qwen72B_Labels.groupby("data_source_corr_trainee")["Match_percent"]

# Function to compute mean, SD, and 95% CI
def mean_sd_ci(series):
    series = series.dropna()
    n = len(series)
    if n == 0:
        return pd.Series({"mean": np.nan, "sd": np.nan, "CI_lower": np.nan, "CI_upper": np.nan, "n": 0})
    mean = series.mean()
    sd = series.std()  # standard deviation
    sem = stats.sem(series)  # standard error
    ci = sem * stats.t.ppf((1 + 0.95) / 2, n-1)  # 95% CI
    return pd.Series({"mean": mean, "sd": sd, "CI_lower": mean - ci, "CI_upper": mean + ci, "n": n})

# Apply function
summary = grouped.apply(mean_sd_ci).reset_index()

# Pivot from long to wide format
summary = summary.pivot(index='data_source_corr_trainee', columns='level_1', values='Match_percent').reset_index()

# Create beautiful formatted table
summary["Mean (%)"] = summary["mean"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["SD (%)"] = summary["sd"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["95% CI"] = summary.apply(
    lambda row: f"[{row['CI_lower']:.2f}, {row['CI_upper']:.2f}]" if not pd.isna(row['CI_lower']) else "N/A",
    axis=1
)
summary["N"] = summary["n"].astype(int)

# Select and rename columns for final display
result = summary[["data_source_corr_trainee", "N", "Mean (%)", "SD (%)", "95% CI"]]
result.columns = ["Data Source", "N", "Mean (%)", "SD (%)", "95% CI"]

result

,Data Source,N,Mean (%),SD (%),95% CI
0,jama,578,69.00,16.78,"[67.63, 70.37]"
1,medbullets,207,68.93,18.71,"[66.36, 71.49]"
2,medxpert,318,69.33,21.33,"[66.98, 71.68]"
3,mmlu,193,72.63,21.22,"[69.62, 75.64]"


In [16]:
import numpy as np
from scipy import stats

# Drop NaNs
all_matches = Qwen72B_Labels["Match_percent"].dropna()

n = len(all_matches)
if n == 0:
    overall_summary = {"N": 0, "Mean (%)": np.nan, "SD (%)": np.nan, "95% CI": (np.nan, np.nan)}
else:
    mean = all_matches.mean()
    sd = all_matches.std()
    sem = stats.sem(all_matches)
    ci = sem * stats.t.ppf((1 + 0.95) / 2, n-1)
    overall_summary = {
        "N": n,
        "Mean (%)": round(mean, 2),
        "SD (%)": round(sd, 2),
        "95% CI": [round(mean - ci, 2), round(mean + ci, 2)]
    }

overall_summary

{'N': 1296,
 'Mean (%)': np.float64(69.61),
 'SD (%)': np.float64(18.99),
 '95% CI': [np.float64(68.57), np.float64(70.64)]}